# Notebook 02 — The Loop

We write the **entire agent in one cell, from scratch**, before we ever import a
framework. Then we see why the loop beats a single shot.

The loop we build is a named, peer-reviewed pattern — **ReAct**
(reason -> act -> observe -> revise, [Yao et al., 2022](https://arxiv.org/abs/2210.03629)).
The hardening we add at the end is **Reflexion** (verbal self-feedback,
[Shinn et al., NeurIPS 2023](https://arxiv.org/abs/2303.11366)). We're hand-building
two real agent patterns, not inventing one.

**Honesty up front:** "closed-loop beats single-shot" is *task- and model-dependent*,
not a law. The win is real only when the task needs grounding in something outside
the model's weights — here, the actual bytes of a CSV. We pick a grounding task on
purpose so the contrast is honest, and we ship a precomputed transcript so the point
survives offline.

## Setup

One line wires the model, the token counter, and the data path. Everything else
in this notebook is the lesson.

In [ ]:
# Shared wiring: the model, an async ask() helper, a token counter, the data path.
from roadshow import setup, Message
from roadshow_viz import compare, lines, timeline
import io, csv, json, contextlib
import numpy as np

model, ask, ntok, DATA = setup()       # provider comes from .env (ARC_PROVIDER) — same as every notebook
PRECOMP = DATA / "precomputed"         # vendored transcripts for the offline path
print("ready · model:", getattr(model, "name", model.__class__.__name__))

## The one tool the loop can call: `run_python`

The agent gets exactly one capability: run a snippet of Python and read back its
stdout/stderr. We show the body so it isn't a black box.

**Security aside (worth saying aloud):** a tool that executes model-written code
is exactly the attack surface security and compliance staff care about. In a real
deployment this `run_python` should be a sandbox — a resource-capped subprocess,
no network, a scratch directory only. The *agent* never runs code itself; it emits
a tool *request*, and the harness decides whether to honor it. We harden this in
the harness notebook.

In [ ]:
# run_python: execute a snippet, capture stdout+stderr as a string observation.
# Pre-bind a few safe names so the model can read our data without imports.
def run_python(code: str) -> str:
    buf = io.StringIO()
    g = {"np": np, "csv": csv, "json": json, "Path": type(DATA),
         "DATA": DATA}
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            exec(code, g)   # demo sandbox stand-in; real deployments isolate this
    except Exception as exc:
        return f"{buf.getvalue()}{type(exc).__name__}: {exc}"
    out = buf.getvalue().strip()
    return out or "(no output)"

TOOLS = {"run_python": run_python}
print("tools:", list(TOOLS))

## ▶ Control cell — the only cell you edit

Everything downstream reads these plain variables. **Change a value, then re-run
the cells below it.** No widgets, no hidden state.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONTROL CELL — edit these, then re-run everything below
# ═══════════════════════════════════════════════════════════════════════════
MODE      = "closed_loop"   # "closed_loop" = the agent loop with tools
                            # "single_shot" = one model call, no tools
MAX_STEPS = 6               # stop after this many loop turns (anti-runaway cap)
TASK      = ("Compute the 90th percentile of the latencies in data/logs/oss04.csv "
             "and say whether it breaches the 50ms SLO.")   # the grounding task
OFFLINE   = False           # True  = replay the vendored transcript (no API calls)
                            # False = run live against the model
VERBOSE   = True            # True  = print the full event stream as the loop runs

print(f"MODE={MODE}  MAX_STEPS={MAX_STEPS}  OFFLINE={OFFLINE}")

## Here is the whole agent

The next cell is the **entire loop**. Four moves, repeated until done:

> **reason -> act -> observe -> revise**

We need three tiny helpers first — a system prompt that tells the model how to
call our one tool, a `parse_tool_call` that reads the model's reply, and a thin
`llm()` wrapper over `model.invoke` (the loop needs raw message lists, so it
talks to the model directly rather than through `ask`). Then the loop itself is
~20 lines.

**The message helpers.** A `ToolCall` record, the system prompt that teaches the model our one-tool protocol, and tiny builders for each message role.

In [ ]:
import re
from dataclasses import dataclass

@dataclass
class ToolCall:
    name: str
    args: dict

def system_prompt(tools: dict) -> Message:
    names = ", ".join(tools)
    return Message(role="system", content=(
        "You are a careful data agent. You may call ONE tool: run_python(code).\n"
        "To call it, reply with EXACTLY one fenced block:\n"
        "```tool\n{\"name\": \"run_python\", \"args\": {\"code\": \"<python>\"}}\n```\n"
        "Make ONE tool call per reply, then wait for the TOOL RESULT before the next.\n"
        "The sandbox has these names PRE-BOUND (do NOT import or pip-install them, "
        "do NOT use subprocess): np (numpy), csv, json, Path, and DATA — a pathlib "
        "Path to the data directory. Read files via DATA / 'subdir' / 'file.csv'. "
        "CSV files have a header row; oss04.csv has a 'latency_ms' column. "
        "The code may print results. When you have the answer, reply in plain prose "
        "with NO fenced block. Available tools: " + names + "."
    ))

def user(task: str) -> Message:        return Message(role="user", content=task)
def assistant(text: str) -> Message:   return Message(role="assistant", content=text)
def tool_result(obs: str) -> Message:  return Message(role="user", content=f"TOOL RESULT:\n{obs}")

**Talk to the model, parse its reply.** `llm()` is a thin wrapper over `model.invoke` (the loop needs raw message lists); `parse_tool_call()` reads the first ```tool block — degrading to `None` (just answer) when a small model emits malformed JSON.

In [ ]:
async def llm(context: list) -> str:
    # The loop passes a growing list of Messages, so it calls the model directly.
    resp = await model.invoke(context)
    return resp.content or ""

def parse_tool_call(text: str):
    # Find the FIRST ```tool ...``` block and parse it. Degrade gracefully -> None.
    m = re.search(r"```tool\s*(\{.*?\})\s*```", text, re.DOTALL)  # noqa: W605
    if not m:
        return None
    try:
        # strict=False: models routinely put raw newlines inside the "code" string,
        # which is illegal in strict JSON. Tolerating them is what makes the loop run.
        d = json.loads(m.group(1), strict=False)
        return ToolCall(name=d["name"], args=d.get("args", {}))
    except Exception:
        return None   # malformed (common on small models) -> loop just answers

**The whole agent — `reason → act → observe → revise`.** ~20 lines. Note `context.append(...)` runs twice per turn: the window *grows* every step (the seed of the runaway later).

In [ ]:
# ── THE WHOLE AGENT ──────────────────────────────────────────────────────────
async def agent(task, tools, max_steps):
    context = [system_prompt(tools), user(task)]
    events = []
    for step in range(max_steps):
        thought = await llm(context)                      # REASON
        events.append(("reason", step, thought))
        call = parse_tool_call(thought)
        if call is None:                                  # REVISE: done?
            events.append(("final", step, thought)); break
        obs = tools[call.name](**call.args)               # ACT
        events.append(("act", step, call))
        events.append(("observe", step, obs))             # OBSERVE
        context.append(assistant(thought))
        context.append(tool_result(obs))                  # the window GROWS
    return events

Point at each move in the code above: **reason** (`llm`), **act**
(`tools[...]()`), **observe** (append the result), **revise** (loop again or
stop). That's the **ReAct** loop ([Yao et al., 2022](https://arxiv.org/abs/2210.03629)).
Everything else today is a refinement of these lines.

Notice `context.append(...)` runs **twice every turn** — the window grows on
every step. That's the seed of the runaway in the failure section, and the
reason **compaction** (CONDENSE, one of Section 1's five context moves) is
mandatory, not optional.

## Run the loop and read the event stream

Run the agent on `TASK` and watch the event stream. With `VERBOSE`, you literally
read the agent think, call `run_python`, see the result, and conclude. With
`OFFLINE=True` it replays the vendored transcript instead of calling the model.

**How we print the event stream** (and how we replay a vendored transcript offline).

In [ ]:
LABELS = {"reason":"💭 reason", "act":"🔧 act   ", "observe":"👁  observe",
          "final":"✅ final "}

def print_events(events):
    for kind, step, payload in events:
        if kind == "act":
            body = f"{payload.name}({list(payload.args)})"
        else:
            body = str(payload).strip().replace("\n", "\n            ")
        print(f"  step {step}  {LABELS.get(kind, kind)} | {body[:300]}")

def load_precomputed_events(path):
    # Reshape a vendored transcript JSON into agent()'s event tuples.
    d = json.load(open(path))
    ev = []
    for e in d["events"]:
        if e["kind"] == "act":
            ev.append(("act", e["step"], ToolCall(e["tool"], e.get("args", {}))))
        else:
            ev.append((e["kind"], e["step"], e["text"]))
    return ev, d

**Run it.** Watch the agent think, call `run_python`, read the result, and conclude.

In [ ]:
if OFFLINE:
    EVENTS, _meta = load_precomputed_events(PRECOMP / "02_healthy_run.json")
    print("(OFFLINE) replaying vendored transcript:\n")
else:
    EVENTS = await agent(TASK, TOOLS, MAX_STEPS)

if VERBOSE:
    print_events(EVENTS)

final = next((p for k, s, p in EVENTS if k == "final"), "(no final answer)")
print("\nFINAL:", str(final).strip())

### Visual — the loop as a timeline

One row per step; a colored segment per move. Read the loop unroll left-to-right within each step.

In [ ]:
timeline(EVENTS, title="ReAct loop: reason → act → observe → revise")

## Open-loop vs closed-loop — the core claim

The loop beats a single shot **because it checks the world**. Same task, two modes:

- `single_shot`: one model call, **no tools**. It can only *guess* a percentile
  from the filename — it never sees the bytes.
- `closed_loop`: the agent runs `run_python` on the actual CSV and gets the real number.

We print both answers next to the ground truth computed directly with numpy.

**Honest framing:** this is NOT "loops are smarter." A single shot literally
*cannot* know a number that lives in a file it was never given — the loop wins
because it has a *path* to the data, not because it reasons better. A well-behaved
model may *refuse to guess* in single-shot (also a fine outcome) — we show it
rather than hiding it.

**Ground truth**, computed directly with numpy — no model involved.

In [ ]:
# Ground truth, computed directly — no model involved.
def true_p90():
    rows = list(csv.DictReader(open(DATA / "logs" / "oss04.csv")))
    v = np.array([float(r["latency_ms"]) for r in rows])
    return round(float(np.percentile(v, 90)), 2)

**Pull the claimed number** out of the model's prose (ignoring the 50ms SLO the prompt itself mentions).

In [ ]:
def _extract_ms(text):
    # Pull the claimed p90 (in ms) out of free text, ignoring the 50ms SLO threshold
    # the prompt itself mentions (otherwise a refusal that echoes "50ms SLO" looks
    # like a confident guess of 50).
    if not text:
        return None
    nums = [(m.start(), float(m.group(1)))
            for m in re.finditer(r"(\d+(?:\.\d+)?)\s*ms", text)]  # noqa: W605
    # drop any "<n> ms SLO/threshold/limit/target" — that's the SLO, not a claim
    nums = [(pos, v) for (pos, v) in nums
            if not re.match(r"\s*(slo|threshold|limit|target)",
                            text[pos:].split("ms", 1)[-1].lower())]
    # No ms-tagged number that isn't the SLO -> the model gave no real claim
    # (e.g. it refused to guess). Returning None shows that honestly on VISUAL 2.
    return nums[0][1] if nums else None

**Solve it both ways** — one mode flag, no duplicated cells: `single_shot` (no tools, must guess) vs `closed_loop` (real tools, real bytes).

In [ ]:
async def solve(task, mode):
    # A/B as a single function over a mode flag (no duplicated cells).
    if OFFLINE:
        d = json.load(open(PRECOMP / "02_ab_pair.json"))
        return {"answer": d[mode]["answer"],
                "claimed_p90": d[mode].get("claimed_p90"),
                "tool_calls": d[mode].get("tool_calls", 0)}
    if mode == "single_shot":
        # one call, NO tools — the model cannot see the file, so any number is a prior.
        answer = await ask(
            task + " You cannot read the file, so estimate from priors and commit to a "
            "single best-guess number, formatted exactly as '<N> ms'.",
            system="You have NO tools and cannot read files. Give your best estimate "
                   "as a single number in ms; do not refuse.")
        return {"answer": answer, "claimed_p90": _extract_ms(answer), "tool_calls": 0}
    # closed loop — real tools, real bytes
    events = await agent(task, TOOLS, MAX_STEPS)
    final = next((p for k, s, p in events if k == "final"), "")
    n_tools = sum(1 for k, *_ in events if k == "act")
    return {"answer": final, "claimed_p90": _extract_ms(final), "tool_calls": n_tools}

**Run both, against the truth.**

In [ ]:
RESULTS = {}
for mode in ["single_shot", "closed_loop"]:
    RESULTS[mode] = await solve(TASK, mode=mode)

TRUTH = true_p90()
print(f"GROUND TRUTH p90 (numpy)        : {TRUTH} ms  ->  {'BREACH' if TRUTH > 50 else 'within'} 50ms SLO\n")
for mode in ["single_shot", "closed_loop"]:
    r = RESULTS[mode]
    print(f"[{mode}]  claimed p90 = {r['claimed_p90']}  tool_calls = {r['tool_calls']}")
    print(f"   {str(r['answer']).strip()[:400]}\n")

### Visual — guess vs grounded vs truth

Only the closed loop lands on the real number. The dashed line is the 50ms SLO.

In [ ]:
# refused-to-guess shows as 0; the printout above has the nuance.
GUESS = {
    "single_shot (guess)":   RESULTS["single_shot"]["claimed_p90"] or 0,
    "closed_loop (computed)": RESULTS["closed_loop"]["claimed_p90"] or 0,
    "numpy truth":            TRUTH,
}
compare(GUESS, title="Only the closed loop lands on the truth",
        ylabel="p90 write latency (ms)", fmt="{:.1f}", hline=(50, "50ms SLO"))

## Now break it — the two classic loop failures

A loop is only as good as its guardrails. We'll induce the two classic *loop*
failures and watch them feed the *context* failure modes from Section 1.

Be precise about vocabulary:

- **Non-termination** and **goal drift** are failures of the loop's **control flow**.
- They are distinct from — but *cause* — Drew Breunig's four **context** failure
  modes: *poisoning, distraction, confusion, clash*
  ([Breunig, 2025](https://www.dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html)).

A runaway loop piles tokens into the window until **distraction** sets in (the model
over-weights its own ballooning transcript) and reliability drops with sheer length —
that's Section 1's **context rot** ([Chroma, 2025](https://research.trychroma.com/context-rot)).
That is the bridge between this notebook and the opening deck.

### Induce non-termination + drift

Give the loop a tool that *always* says "not yet", a task with no terminal
condition, and a big step cap. Watch it spin: the goal is never re-asserted, so
it wanders and never stops. We track the context size each step. The 50-step live
spin burns tokens, so `OFFLINE=True` replays a vendored runaway transcript instead.

**A tool that never says "done."** `check_status()` always returns *not yet* — so the loop has no terminal condition.

In [ ]:
def check_status() -> str:
    return "not yet"          # never returns a terminal state -> non-termination

RUNAWAY_TASK = "Wait until batch job 8841 has finished, then report its exit code."
RUNAWAY_MAX_STEPS = 50

**Run it** (or replay the vendored runaway transcript). Watch the window balloon.

In [ ]:
if OFFLINE:
    runaway = json.load(open(PRECOMP / "02_runaway.json"))
    steps_taken = len(runaway["cumulative_tokens"])
    tokens_per_step = runaway["cumulative_tokens"]
    terminated = runaway["terminated"]
    print("(OFFLINE) vendored runaway transcript:")
    print(" ", runaway["reason"])
else:
    # Live: poll the never-terminating tool; estimate context growth as chars/4.
    poll_tools = {"check_status": lambda: check_status()}
    context = [Message(role="system", content=(
                   "You may call check_status() by replying with ```tool\n"
                   "{\"name\": \"check_status\", \"args\": {}}\n```. "
                   "Report the exit code once the job has finished.")),
               Message(role="user", content=RUNAWAY_TASK)]
    tokens_per_step, terminated = [], False
    cum = sum(len(m.content) for m in context) // 4
    for step in range(RUNAWAY_MAX_STEPS):
        thought = await llm(context)
        call = parse_tool_call(thought)
        if call is None:
            terminated = True
            tokens_per_step.append({"step": step, "cum_tokens": cum + len(thought)//4})
            break
        obs = poll_tools[call.name]()
        context.append(assistant(thought)); context.append(tool_result(obs))
        cum = sum(len(m.content) for m in context) // 4
        tokens_per_step.append({"step": step, "cum_tokens": cum})
    steps_taken = len(tokens_per_step)

print(f"\nsteps taken: {steps_taken} / {RUNAWAY_MAX_STEPS}   terminated cleanly? {terminated}")
print(f"final window size: ~{tokens_per_step[-1]['cum_tokens']} tokens "
      f"(started ~{tokens_per_step[0]['cum_tokens']})")
print("Diagnosis: no terminal condition + goal never re-asserted -> non-termination + drift -> distraction.")

### Visual — token growth: runaway vs healthy

Context is a *budget, not a bucket*. Degradation begins well below the hard window limit (the dashed line).

In [ ]:
runaway_curve = [d["cum_tokens"] for d in tokens_per_step]
curves = json.load(open(PRECOMP / "02_token_curves.json"))
healthy_curve = [d["cum_tokens"] for d in curves["healthy"]]
lines({"runaway loop": runaway_curve, "healthy loop (terminates)": healthy_curve},
      title="Context is a budget, not a bucket", xlabel="loop step",
      ylabel="cumulative context (~tokens)", hline=(curves["budget"], "context-rot zone"))

## Add the guardrails

Three fixes, each one line, each killing a specific failure:

| guardrail | one-liner | kills |
|---|---|---|
| **step cap** | `for step in range(max_steps)` | non-termination |
| **goal re-assertion** | re-inject the goal every turn | drift / distraction (the **Reflexion** move, [Shinn et al., 2023](https://arxiv.org/abs/2303.11366)) |
| **history compaction** | summarize older turns over a token budget | token blowup / context rot (**CONDENSE** from Section 1) |

### The hardened loop

Same loop, three guardrails added inline — run against the **exact task that ran
away for 50 steps above**. The step cap guarantees it stops within `MAX_STEPS`,
goal re-assertion lets the model decide the job is stuck and stop on its own, and
compaction summarizes old turns when the window crosses a budget. Watch the step
count collapse from 50 and the window stay bounded; the compaction events print below.

**Estimate the window size** (~chars / 4).

In [ ]:
def estimate_tokens(messages):
    return sum(len(m.content) for m in messages) // 4

**Compaction (CONDENSE).** Over budget, replace older turns with a one-line summary; keep system + goal + the most recent turn.

In [ ]:
def compact(context, goal, budget):
    # CONDENSE: replace older turns with a one-line summary; keep system+goal+recent.
    if estimate_tokens(context) <= budget:
        return context, None
    head = context[:1]                       # system prompt
    recent = context[-2:]                     # most recent turn
    middle = context[1:-2]
    summary = Message(role="user", content=(
        f"[COMPACTED {len(middle)} earlier messages] "
        f"Progress so far: polled status, no terminal result yet. Goal still open."))
    return head + [summary] + recent, f"compacted {len(middle)} msgs"

# Same never-terminating scenario as the runaway, now WITH the three guardrails.
HARD_SYS = ("You may call check_status() by replying with ```tool\n"
            "{\"name\": \"check_status\", \"args\": {}}\n```. "
            "Report the exit code once the job has finished.")

**The hardened agent** — the same loop, now with three guardrails: a step cap, goal re-assertion every turn, and compaction on budget.

In [ ]:
async def hardened_agent(task, tools, max_steps, token_budget=600):
    context = [Message(role="system", content=HARD_SYS), user(task)]
    events, compactions = [], []
    for step in range(max_steps):                         # GUARDRAIL 1: step cap
        # GUARDRAIL 2: re-assert the goal every turn (anti-drift / Reflexion)
        context.append(Message(role="user", content=f"REMINDER — your goal: {task}"))
        thought = await llm(context) if not OFFLINE else "Job still 'not yet'; stopping."
        events.append(("reason", step, thought))
        call = parse_tool_call(thought) if not OFFLINE else None
        if call is None:
            events.append(("final", step, thought)); break
        obs = tools[call.name](**call.args)
        events.append(("act", step, call)); events.append(("observe", step, obs))
        context.append(assistant(thought)); context.append(tool_result(obs))
        # GUARDRAIL 3: compaction (anti-blowup / CONDENSE) — keeps the window bounded
        context, note = compact(context, task, token_budget)
        if note:
            compactions.append((step, note))
            events.append(("compact", step, note))
    return events, compactions, estimate_tokens(context)

**Run the SAME runaway task, now bounded.**

In [ ]:
# The SAME task that ran away for 50 steps above — now bounded by the guardrails.
HARD_TASK = RUNAWAY_TASK
HARD_TOOLS = {"check_status": lambda: check_status()}

hard_events, compactions, final_window = await hardened_agent(HARD_TASK, HARD_TOOLS, max_steps=MAX_STEPS)
stopped_early = any(k == "final" for k, *_ in hard_events)
steps_used = max(s for _, s, _ in hard_events) + 1
print(f"runaway above: 50 / 50 steps, never stopped.")
print(f"hardened here: {steps_used} / {MAX_STEPS} steps  "
      f"({'stopped on its own' if stopped_early else 'cut off by the step cap'}).")
print(f"final window: ~{final_window} tokens (compaction budget 600) — bounded, not ballooning.")
print("compaction events:", compactions or "(none needed under budget)")

### Visual — compaction before/after

Compaction lets the loop survive its own memory: the compacted curve sawtooths under budget instead of climbing forever.

In [ ]:
compacted_curve = [d["cum_tokens"] for d in curves["compacted"]]
lines({"runaway (no compaction)": runaway_curve, "compacted (CONDENSE on budget)": compacted_curve},
      title="Compaction lets the loop survive its own memory", xlabel="loop step",
      ylabel="cumulative context (~tokens)", hline=(curves["budget"], "token budget"))

## One loop, every framework

You just hand-built what `arcrun.run()` gives you. The same task through the real
framework has the **same event stream, same shape** — just hardened and observable
out of the box.

The Section-1 callback, stated plainly: the guardrails you just added *are* the
harness, and a good harness is worth more than a model upgrade — agent scaffolding
has produced **20+ point swings on SWE-bench on identical weights**
([Particula, 2025](https://particula.tech/blog/agent-scaffolding-beats-model-upgrades-swe-bench)).
Phrase it exactly: **good harness + decent model > weak harness + frontier model**.
For an on-prem deployment that's the thesis: you don't need the biggest model, you
need the loop around it to be sound.

Run the same task through `arcrun.run()` and compare. The 2nd positional arg is
a capability *provider* (`StaticProvider` wraps our tool); the system prompt and
task are positional. Watch the result come back with turns, tool calls, and cost
already measured.

In [ ]:
from arcrun import run, Tool, ToolContext, StaticProvider

# The same run_python capability, wrapped as a framework Tool.
async def run_python_tool_fn(args: dict, ctx: ToolContext) -> str:
    return run_python(args["code"])

run_python_tool = Tool(
    name="run_python",
    description=("Execute a short Python snippet and return its stdout/stderr. "
                 "PRE-BOUND names (do not import / pip-install / use subprocess): "
                 "np (numpy), csv, json, Path, and DATA (a pathlib Path to the data "
                 "dir). Read files via DATA / 'subdir' / 'file.csv'."),
    input_schema={
        "type": "object",
        "properties": {"code": {"type": "string", "description": "Python to execute"}},
        "required": ["code"],
    },
    execute=run_python_tool_fn,
)

if OFFLINE:
    d = json.load(open(PRECOMP / "02_healthy_run.json"))
    print("(OFFLINE) arcrun.run() would produce the same shape:")
    print("  final:", d["events"][-1]["text"])
    print("  turns: 2  tool_calls: 1  (same loop, hardened + observable)")
else:
    result = await run(
        model,
        StaticProvider([run_python_tool]),
        ("You are a careful data agent. Use run_python to compute on local files. "
         "The sandbox has DATA pre-bound (a pathlib Path to the data dir) plus np, "
         "csv, json, Path. Read files via DATA / 'subdir' / 'file.csv' — do not "
         "import, pip-install, or use subprocess."),
        TASK,
        max_turns=MAX_STEPS,
    )
    print(result.content)
    cost = f"${result.cost_usd:.4f}" if result.cost_usd is not None else "n/a"
    print(f"\nturns: {result.turns}  tool_calls: {result.tool_calls_made}  cost: {cost}")
    print("\nSame reason->act->observe->revise loop you hand-built — now hardened and observable.")

## ✅ TODO — give the loop your own tool

Write a **second** tool and a task that needs **two** tool calls in sequence, so
you feel the loop *chain* them. The model plans the order — you don't script it.
Fill in the `read_file` tool below, then run a task that forces `read_file` then
`run_python`.

In [ ]:
# ✅ TODO — add a second tool, then run a task that needs BOTH in sequence.

async def read_file_tool_fn(args: dict, ctx: ToolContext) -> str:
    # TODO: read up to 2000 chars from args["path"] (resolve against DATA's parent),
    #       and return "Path not found: ..." if it doesn't exist.
    return "TODO"

read_file_tool = Tool(
    name="read_file",
    description="Read up to 2000 chars from a text file.",
    input_schema={
        "type": "object",
        # TODO: describe the "path" property and mark it required.
        "properties": {},
        "required": [],
    },
    execute=read_file_tool_fn,
)

# Once your tool works, run a task that forces read_file THEN run_python.
# Same call shape as the cell above: run(model, StaticProvider([...]), system_prompt, task).
# result = await run(
#     model,
#     StaticProvider([read_file_tool, run_python_tool]),
#     "You are a careful data agent. Use tools.",
#     ("Read data/incidents/storage_incident_2026-04-22.md, then compute the p90 "
#      "of data/logs/oss04.csv and say whether it explains the incident's latency spike."),
#     max_turns=MAX_STEPS,
# )
# print(result.content)   # watch it chain: read -> compute -> answer

## Takeaway

> ✅ **Takeaway:** An agent is a loop that **reasons, acts, observes, and revises**
> (ReAct) while **managing its own context window** — and it only earns its keep when
> guardrails (**stop cap**, **goal re-assertion**, **compaction**) keep it from running
> away. Everything from here is about giving that loop better equipment; the first
> piece is the harness.

**Next:** [03 — The Harness →](./03_harness.ipynb)